[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/agentic-systems-foundations-design/blob/main/notebooks/04_agent_skills.ipynb)


# Agentic Systems Foundations
## Notebook 04: Agent Skills — Composable Competence
**Duration:** 40 min &nbsp;|&nbsp; **Mode:** Demonstration + Guided Practice

> Taught **WHY → WHAT → HOW**. The recurring question all session is
> ***"what does this step look like when it goes wrong — and where would
> you see it in the trace?"*** We **predict before we run** and **compare
> outputs**. LangChain appears as a **parallel mapping**, never as the
> primary path.

![loop](https://dummyimage.com/1000x64/1f2937/ffffff&text=GOAL+%E2%86%92+STATE+%E2%86%92+THINK+%E2%86%92+ACT+%E2%86%92+OBSERVE+%E2%86%92+%5Bterminate%3F%5D+%E2%86%92+ANSWER)

**Where we are in the loop:** **THINK**, again — but now we control *what the agent is even allowed to consider*.

> **Requires `OPENAI_API_KEY`.** These notebooks call a real model —
> there is no simulated fallback, on purpose.


In [ ]:
# ============================================================
# BOOTSTRAP — run this cell first. (Same in every notebook.)
# ============================================================
# Works in Colab, a local venv, or a bare Jupyter. It installs whatever is
# actually MISSING rather than assuming a particular environment — checking by
# import is the only reliable test.
import importlib.util, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/agentic-systems-foundations-design.git"   # INSTRUCTOR: change this if you fork

# import name -> pip package name
REQUIRED = {
    "openai": "openai",
    "dotenv": "python-dotenv",
    "jsonschema": "jsonschema",
    "langchain_core": "langchain-core",
    "langchain_openai": "langchain-openai",
    "langgraph": "langgraph",
}

def _present(module: str) -> bool:
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError, ModuleNotFoundError):
        return False

missing = sorted({pkg for mod, pkg in REQUIRED.items() if not _present(mod)})
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
else:
    print("dependencies: all present")

try:
    import agent_core
except ModuleNotFoundError:
    # Try a clone (Colab), then a parent directory (local repo checkout).
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    for candidate in ("agentic-systems-foundations-design", "..", "."):
        if os.path.isdir(os.path.join(candidate, "agent_core")):
            sys.path.insert(0, os.path.abspath(candidate))
            break
    import agent_core

from pathlib import Path
_pkg = Path(agent_core.__file__).resolve().parent
print("agent_core", agent_core.__version__, "from", _pkg)
print("data dir  :", _pkg.parent / "data", "(exists:", (_pkg.parent / "data").exists(), ")")

In [ ]:
# ============================================================
# YOUR API KEY  (required — there is no offline fallback)
# ============================================================
# Every notebook in this session calls a REAL model. There is deliberately no
# simulated fallback: a fake model can show you the shape of an agent loop, but
# it cannot show you how a real one behaves when your tool descriptions are
# ambiguous or your schema is too loose — and that behaviour is the subject of
# the session.
#
#   Colab : sidebar -> key icon -> add a secret named OPENAI_API_KEY
#           -> toggle "Notebook access" ON -> re-run this cell
#   local : export OPENAI_API_KEY=sk-...   (or put it in a .env file)
import os

def _load_key() -> bool:
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass
    return False

if not _load_key():
    raise RuntimeError(
        "OPENAI_API_KEY is not set — this notebook calls a real model.\n"
        "Colab: sidebar -> key icon -> add OPENAI_API_KEY -> Notebook access ON.\n"
        "Local: export OPENAI_API_KEY=sk-...  then restart the kernel."
    )

from agent_core import current_config, get_llm
llm = get_llm()
print(current_config())
print("active LLM:", llm.name)
print()
print("These notebooks spend real tokens. Budgets are deliberately small.")

## WHY — the problem you only meet at tool number twelve

Your refund agent works. Now the business asks for billing disputes. Then
onboarding help. Then incident triage.

The obvious move is to keep adding: more tools to the list, more paragraphs to
the system prompt. It works, until it doesn't — and it fails in two ways at
once, both of them quiet:

- **Tool choice degrades.** Every tool you add is one more wrong option. At five
  tools the model picks well. At twenty-five it picks *plausibly*. There is no
  error; there is a reasonable-looking call to the wrong tool.
- **The prompt becomes unownable.** Instructions for four unrelated jobs sit in
  one block of text. They start contradicting each other. Nobody can change the
  refund wording without risking the onboarding behaviour.

> **The recurring question, for this step:** an over-scoped agent shows up in the
> trace as a *successful* call to a tool that had nothing to do with the goal.
> Everything is green. The answer is wrong.


## WHAT — a skill is a job, scoped

```
Skill  =  instructions  +  a scoped tool subset  +  a termination policy
```

```python
Skill(name="refunds",
      instructions="You handle refund requests. Look up the order first…",
      tools=registry.subset("get_order_status", "check_refund_eligibility",
                            "search_docs", "escalate_to_human"))
```

The **scoping is the load-bearing part.** A skill with good instructions but the
full tool list is just a prompt template — the model can still reach for the
wrong tool, so you do not get the reliability improvement.

### Two ways to combine skills, and knowing which is the actual skill

| | What it does | When |
|---|---|---|
| **`Skill.compose(a, b)`** | one skill, union of tools, joined instructions | a single request genuinely spans both jobs |
| **`Router`** | picks exactly ONE skill per request | requests are separable — **usually** |

Compose *undoes* the scoping benefit: a composed skill has all the tools of its
parts, which is the situation skills existed to prevent. **Route by default;
compose when you must.**

Routing is how a system reaches twenty-five tools while no single agent ever
sees more than four.


## HOW — look at the scoping, then watch it route

In [ ]:
from agent_core.acme_tools import acme_registry
from agent_core.skills import acme_skills, acme_router

registry = acme_registry()
skills = acme_skills(registry)

print(f"FULL TOOLBOX ({len(registry)} tools): {', '.join(registry.names())}\n")
for skill in skills:
    print(f"{skill.name:<20} {len(skill.tools)} tools: {', '.join(skill.tools.names())}")

print("\nNote: no skill gets all five. Note also the deliberate OVERLAP —")
print("search_docs and escalate_to_human each appear in two skills.")
print("Skills are NOT a partition of your tools. The question for each skill is")
print("'what does THIS job need?', asked independently.")

In [ ]:
# Routing, with its reasoning shown. A router you cannot inspect is a router
# you cannot debug — and "the agent did the wrong job" is nearly always a
# routing bug, not a loop bug.
router = acme_router(registry)

for goal in [
    "How much does the Growth plan cost per month?",
    "I want a refund on ACME-1046, I changed my mind.",
    "What is the status of order ACME-1048?",
    "What is 199 multiplied by 12?",
]:
    print(router.explain(goal))
    print()

Look at the last one: *"What is 199 multiplied by 12?"* scores **0 everywhere**.
No trigger matches. So the router falls back — and *which* skill it falls back to
is a configuration choice we made explicitly:

```python
Router(skills, fallback=product_questions)   # the general-purpose skill
```

`Router` defaults to `skills[0]`. Leave the fallback unset and an unmatched
request lands in whichever skill happens to be listed first — an agent answering
an arithmetic question with the refunds toolbox. Not a loop bug, not a model bug:
a one-line configuration bug, invisible until you print the routing decision.


> ### ✋ Predict before you run
> Next we run the same six goals two ways: through the **Router** (each request sees ~3 tools) and through one **composed mega-skill** (every request sees all 5). **Will the mega-skill call more tools, fewer, or the same?** And will it get more or fewer of them right?
>
> *Commit to a guess before executing. Comparing your prediction with the result is where the learning actually happens.*


In [ ]:
# ROUTED (scoped) vs COMPOSED (everything at once) — the core comparison.
from agent_core import Agent, Skill

routed_agent   = Agent(skill=acme_router(registry))
mega = Skill.compose(*skills, name="everything")
composed_agent = Agent(skill=mega)

print(f"mega-skill sees {len(mega.tools)} tools: {', '.join(mega.tools.names())}\n")

goals = [
    "How much does the Growth plan cost per month?",
    "What is the status of order ACME-1048?",
    "What is 199 multiplied by 12?",
    "I want a refund on ACME-1046, I changed my mind.",
    "What does the refund policy say about orders still processing?",
    "Has order ACME-1043 shipped yet?",
]

print(f"{'goal':<46} {'ROUTED':<34} COMPOSED")
print("-" * 112)
routed_calls = composed_calls = 0
for goal in goals:
    r = routed_agent.run(goal)
    c = composed_agent.run(goal)
    routed_calls += r.trace.tool_calls()
    composed_calls += c.trace.tool_calls()
    print(f"{goal[:44]:<46} {'→'.join(r.tools_called())[:32]:<34} {'→'.join(c.tools_called())[:40]}")

print("-" * 112)
print(f"{'TOTAL TOOL CALLS':<46} {routed_calls:<34} {composed_calls}")

**What you should observe:** the composed agent makes **more tool calls** for the
same goals. With everything in scope it reaches for tools the job did not need.

Every one of those extra calls is real: an LLM round-trip, latency, tokens added
to the transcript for every subsequent step, and one more chance to observe
something irrelevant and reason from it.

Nothing errored. The suite would look healthy. That is what makes over-scoping a
*quiet* failure — and quiet failures are the ones that reach production.

> **Scoping is a reliability technique, not tidiness.**


## HOW — inside a skill: what the model actually receives

In [ ]:
# A skill's system prompt = shared rules + the job + the tool inventory.
refunds = next(s for s in skills if s.name == "refunds")
print(refunds.system_prompt())

Two things to notice in that prompt.

**First**, `BASE_INSTRUCTIONS` is inherited by every skill. Each of its five
rules exists because omitting it produces a *specific* failure we will meet in
notebook 06 — rule 4 ("if a tool says NOT FOUND, that is the answer") is the one
standing between you and an invented order.

**Second**, the tools are listed in the prompt *as well as* being sent as schemas
through the API. That is not redundant. The schema describes the **shape** of a
call; only the prompt describes the **judgement** — when to prefer this tool over
that one. Shape is what the model gets right anyway. Judgement is what it gets
wrong.


## HOW — build a skill (guided practice)

In [ ]:
# YOUR TURN. Build a fourth skill: "usage_and_billing".
#
# It should answer questions about quota and overage charges. Decide:
#   1. which of the existing tools it needs — and be strict
#   2. instructions that say what to do FIRST
#   3. triggers specific enough to win its own cases and lose everyone else's
from agent_core import Skill, Router

usage_skill = Skill(
    name="usage_and_billing",
    instructions=(
        "Answer questions about quota usage and overage charges. Look up the "
        "order first to establish which plan applies, then find the plan's "
        "limits and overage rates in the documentation, then use calculate for "
        "any charge arithmetic. Never estimate a charge yourself."
    ),
    tools=registry.subset("get_order_status", "search_docs", "calculate"),
    # PHRASES, not bare words — see the routing note below.
    triggers=[r"overage", r"quota", r"usage", r"over the limit", r"api calls"],
)

extended = Router(skills + [usage_skill],
                  fallback=next(s for s in skills if s.name == "product_questions"))

for goal in ["Am I over my API call quota on ACME-1042?",
             "What is the status of order ACME-1048?",
             "I want a refund on ACME-1046, I changed my mind."]:
    print(f"{extended.route(goal).name:<20} <- {goal}")

### The routing bug you will write at least once

Open `agent_core/skills.py` and read the comment on `account_lookup`'s triggers.

An earlier version of this package triggered that skill on the bare words
`"order"` and `"status"`. It looked sensible. It quietly stole **every refund
request**, because refund requests mention an order too — and the agent then did
a competent job of the wrong task.

Narrowing it to only shipping words broke its own core case instead. The fix was
**phrases**: `"status of"` is specific enough to win a status question and
specific enough to lose to `"refund"` when the request is really about a refund.

> **The general rule:** a bare-word trigger on a general skill outranks a precise
> trigger on a specific one. The symptom is a competent agent confidently doing
> the wrong job — and you will only find it by printing the routing decision.


In [ ]:
# See it: run the same goal through a deliberately over-broad router.
broken = Router([
    Skill(name="greedy_lookup",
          instructions="Look up orders.",
          tools=registry.subset("get_order_status"),
          triggers=[r"order", r"status"]),          # BARE WORDS — too greedy
    refunds,
], fallback=refunds)

goal = "Check the status of order ACME-1043 and tell me if I can get a refund for changed_mind."
print("OVER-BROAD triggers:", broken.route(goal).name, " <- wrong job")
print("PHRASE triggers    :", acme_router(registry).route(goal).name, " <- correct")
print()
print(broken.explain(goal))

## HOW (parallel mapping) — skills and supervisors in LangGraph

| We build | LangGraph | OpenAI Agents SDK |
|---|---|---|
| `Skill` | a scoped subgraph with its own tools | an `Agent` with `instructions` + `tools` |
| `Router` | a **supervisor** node | agent **handoffs** |
| `Skill.compose` | one graph with a merged tool list | one agent with more tools |

The names differ; the design decision does not. **Every framework's answer to
"we have too many tools" is: give each job its own scoped agent and route between
them.**

The one thing production does differently is *how* it routes: not keyword
scoring, but a cheap model with **structured output**.


In [ ]:
# ============================================================
# LANGGRAPH TRACK
# ============================================================
# The same model, reached through LangChain rather than the OpenAI SDK directly.
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model=os.getenv("AGENT_LLM_MODEL", "gpt-4o-mini"), temperature=0)
print("LangGraph track ready ->", chat.model_name)

In [ ]:
# Keyword routing works identically in the LangGraph track. Pure Python.
# Same skills, same scoping, same phrase-trigger discipline.
from agent_lc import KeywordSupervisor, acme_lc_skills

lc_skills = acme_lc_skills()
for s in lc_skills:
    print(f"{s.name:<20} {len(s.tools)} tools: {', '.join(t.name for t in s.tools)}")

sup = KeywordSupervisor(lc_skills)
print()
print(sup.explain("I want a refund on ACME-1046, I changed my mind."))

In [ ]:
# Production routing: a cheap model classifies, constrained by structured output.
#
# Note WHY structured output rather than 'reply with the skill name': parsing a
# skill name out of prose is a classic source of routing flakiness. Constrain
# the model to a Literal of real skill names and that failure disappears.
from langchain_core.messages import HumanMessage
from agent_lc import build_supervisor_graph, call_sequence, final_answer

supervisor = build_supervisor_graph(chat)

for goal in ["I want a refund on ACME-1046, I changed my mind.",
             "How much does the Growth plan cost per month?"]:
    out = supervisor.invoke({"messages": [HumanMessage(goal)],
                             "steps": 0, "stop_reason": None, "skill": None})
    print(f"{out['skill']:<20} {' -> '.join(call_sequence(out)) or '(none)'}")
    print(f"   {out['stop_reason']}")

**What to notice:** the executing agent still sees **only its own skill's
tools**. That is the property that buys the reliability, and neither routing
strategy changes it — swapping a keyword scorer for an LLM classifier improves
*which* skill gets picked, not what happens after.

**What LangGraph adds that we did not have:** real **handoffs**. Because skills
are nodes in a graph, a skill can route *onward* to another and the state carries
across. That is the honest answer to "what if a request spans two jobs?" — at
scale you neither compose everything nor accept a half-answer; you hand off.


## Recap

- A **skill** = instructions + **scoped tools** + a stop policy. The scoping is
  the part that buys reliability.
- Over-scoping fails **quietly**: successful calls to tools the job never needed.
- **Route by default, compose when a single request genuinely spans two jobs.**
- Set your fallback skill explicitly, or unmatched requests land wherever.
- Broad triggers steal requests from specific skills. Use phrases; print the
  routing decision before blaming the loop.

**Next → Notebook 05 (Control & Tracing):** the agent is capable and well-scoped.
Now — how does it *stop*, and how would you ever know what it did?
